# Retail Demand Forecasting — Full Story
### M5 Walmart Sales Dataset

---

This notebook tells the complete story of the project from business problem through results.
Every modeling decision is explained, every result is interpreted, and every chart connects
back to a concrete business implication.

**Charts live in:** `outputs/presentation_charts/`

In [ ]:
from IPython.display import Image, display
import os

CHART_DIR = '../outputs/presentation_charts'

def show(filename, width=1000):
    path = os.path.join(CHART_DIR, filename)
    display(Image(filename=path, width=width))

---
---

# PART 1 — THE PROBLEM

---

## 1.1 — What does Walmart actually need?

Walmart runs over 10,000 stores. Every store carries thousands of products.
Every one of those products needs to be on the shelf at the right time — not too much, not too little.

**If you order too much:** product expires, takes up shelf space, gets marked down at a loss.

**If you order too little:** the shelf is empty when a customer wants to buy. This is called a **stockout**.
You lose the sale, and possibly the customer permanently.

This problem — predicting how much to order — is called **demand forecasting**.

---

### Key definitions

| Term | Definition |
|---|---|
| **Demand forecasting** | Predicting how many units of a specific product a specific store will sell in a future time period |
| **Stockout** | A product is out of stock when a customer wants to buy it — lost sale |
| **Overstock** | Too much inventory ordered — leads to waste, markdowns, or expired product |
| **Time series** | A sequence of values recorded at regular time intervals (daily sales, monthly revenue) |
| **Hierarchy** | Forecasts can be made at different levels of aggregation — platform total, store, department, individual product |

---

### The business question

> *Given a product's price history, upcoming holidays, local events, and past sales —*
> *how many units will a specific store sell next week?*
> *And are sudden sales spikes genuine demand, or anomalies we should flag?*

The hard part: you are not forecasting one product. You are forecasting **30,490 product-store
combinations simultaneously**, each with its own demand pattern, price history, and local context.

---

## 1.2 — The dataset: M5 Walmart

We use the **M5 Forecasting dataset** — a real Walmart sales dataset released for a Kaggle
competition in 2020. This is not simulated data. These are actual Walmart transactions.

### Scale

| Dimension | Value |
|---|---|
| Unique product-store series | 30,490 |
| Unique products | 3,049 |
| Stores | 10 (across CA, TX, WI) |
| Date range | Jan 29 2011 → Apr 24 2016 (5.25 years) |
| Daily observations | 1,913 days |
| Total rows (long format) | 58,327,370 |

### Three source files

| File | What it contains | How we use it |
|---|---|---|
| `sales_train_validation.csv` | Units sold per product-store per day (wide format) | Primary target — reshaped to long format |
| `calendar.csv` | Dates, day-of-week, events, holidays, SNAP flags | Join to get real dates and external demand drivers |
| `sell_prices.csv` | Weekly price per product per store | Join to compute revenue = units × price |

### Important: revenue is derived, not observed

We compute revenue as `units_sold × sell_price`. The price join introduces minor gaps
for product-store-week combinations with no price record — these are filled with zero,
slightly understating revenue for products with incomplete price histories.

### Wide format vs long format

The sales file arrives in **wide format** — one row per product, one column per day (1,913 columns).
This is completely unusable for time series analysis. We reshape it to **long format** — one row
per product-store-day — before any analysis is possible. This produces 58 million rows.

---

## 1.3 — Why this dataset is hard: zero-inflation

The first thing you discover when you look at the data is that most product-store-days
have **zero sales**.

**68.2% of all product-store-days have zero units sold.**

If you pick any specific product at any specific store on any given day, there is a 68% chance
it sold nothing that day. This is not a data quality issue — it is the nature of retail demand
at the individual series level.

### Two types of zeros — this distinction matters

| Type | Description | Example |
|---|---|---|
| **Intermittent demand zeros** | The product exists and is stocked, but just didn't sell that day | A slow-moving hobby item that sells a few times a week |
| **Structural zeros** | The product wasn't stocked at that store yet, or was discontinued | A new product that launched in 2014 has zeros for all of 2011–2013 |

**Mean longest zero streak per series: 430 consecutive days.**
44% of series have a streak longer than 365 days — meaning a specific product at a specific
store recorded zero sales every single day for over a year.

### Why this matters for modeling

You cannot use lag features ("sales from 4 weeks ago") that reach back across a structural
zero gap — that would be pulling signal from a period when the product didn't exist at that
store, which produces meaningless or misleading values.

In [ ]:
show('02_zero_inflation.png')

**Reading this chart:** The distribution of units sold on non-zero days is extremely right-skewed.
The median is 2 units, the 95th percentile is modest, but the tail extends to 763 units in a
single day at one store. Most individual series sell very little on most days —
the signal lives in the aggregate.

---
---

# PART 2 — EXPLORATORY DATA ANALYSIS

---

## EDA philosophy: every finding drives a decision

EDA is not just exploring for curiosity. Every finding below directly determined
a modeling choice — which model to use, which features to engineer, which series
to model, and how to interpret results. We will flag the decision each finding drives.

---

## 2.1 — Finding 1: Revenue structure — where the money is

Before modeling anything we need to understand which products and stores
carry the most signal. Not all series are worth modeling equally.

In [ ]:
show('03_revenue_by_category.png')

### Key findings

| Finding | Value |
|---|---|
| FOODS share of total revenue | 58% ($108.9M) |
| FOODS_3 alone | 37.8% — single largest department |
| CA_3 (highest revenue store) | 17.1% of platform revenue |
| CA_4 (lowest revenue store) | 6.5% — in the same state as CA_3 |
| HOBBIES_2 | 0.63% — essentially no signal |

**CA_3 is nearly 3× CA_4 despite being in the same state.** State alone is not
a sufficient feature — store-level encoding is essential.

### Modeling decision
Our representative individual series comes from FOODS_3 at CA_3 — the highest-revenue
department at the highest-revenue store. This gives us the most signal for individual-level
modeling. HOBBIES_2 is too sparse to be useful.

---

## 2.2 — Finding 2: Trend and seasonality in the aggregate series

The aggregate monthly revenue series — all products, all stores, summed — is the
foundation for our platform-level SARIMA and Prophet models. Before choosing any
model parameters we need to understand its structure.

### What is seasonal decomposition?

Any time series can be mathematically separated into three components:

| Component | What it is | What it looks like |
|---|---|---|
| **Trend** | The underlying direction of the series — up, down, or flat | A smooth curve rising or falling over years |
| **Seasonality** | The repeating calendar pattern — same every 12 months | Regular peaks and troughs at the same time each year |
| **Residual** | Everything left over after removing trend and seasonality | Random-looking noise — promotions, weather, anomalies |

We use an **additive decomposition**: `observed = trend + seasonal + residual`.
This assumes the seasonal amplitude (the size of the peaks and troughs) stays
constant regardless of the trend level.

In [ ]:
show('04_seasonal_decomposition.png')

### Reading this chart

**Observed (top):** Raw monthly revenue — you can see both the upward trend and
slight waviness but it is hard to separate them visually.

**Trend:** Clean, steady climb from ~$2M/month in 2011 to ~$4M/month by 2016.
Revenue roughly doubled over 5 years. No reversals. NaNs at the edges are normal —
additive decomposition cannot estimate trend at the boundaries of the series.

**Seasonal:** The repeating pattern. August and March are the strongest positive
months (+$147K and +$128K above baseline). February and November are weakest
(-$152K and -$149K). These effects are **small relative to the trend** —
seasonality is a secondary signal here, not the dominant one.

**Residual:** Everything the model cannot explain structurally. Well-behaved —
looks like random noise with no obvious pattern. This is what a good model leaves behind.

### Modeling decision
The trend is strong and persistent. Any model that mean-reverts (pulls forecasts
back toward the historical average at long horizons) will underperform. This directly
motivates why Prophet outperforms SARIMA on this series — we will return to this.

---

## 2.3 — Finding 3: SNAP effects

The calendar file contains SNAP flags — one of the most valuable features in the dataset.

### What is SNAP?

**SNAP (Supplemental Nutrition Assistance Program)** is the US government food assistance
program. Benefits are distributed to eligible households on specific days each month.
When SNAP benefits are distributed, households spend them immediately — producing a
predictable demand spike at grocery stores.

**Critically: each state has a different SNAP distribution schedule.** California, Texas,
and Wisconsin each have their own calendar of SNAP days. A single global flag would
miss the state-specific timing.

In [ ]:
show('05_snap_effect.png')

### SNAP uplift results

| State | All categories | FOODS only |
|---|---|---|
| Wisconsin | +20.5% | +32.5% |
| Texas | +10.8% | +17.2% |
| California | +7.2% | +10.3% |

The effect nearly doubles when isolating FOODS — which makes sense, since SNAP
benefits can only be spent on food items. Wisconsin's effect is the strongest
because it has a smaller store footprint and a higher proportion of SNAP-dependent households.

### Modeling decision
SNAP flags must be **state-level features**: `snap_CA`, `snap_TX`, `snap_WI` as
three separate binary inputs in XGBoost. A single `is_snap_day` flag would lose
the different magnitudes per state and be significantly less useful.

---

## 2.4 — Finding 4: Price elasticity

Prices change week over week in this dataset. Understanding how price changes
affect demand tells us which price features to engineer — and at what level
of granularity they carry real signal.

### What is price elasticity?

**Price elasticity** measures how sensitive demand is to a change in price.
If a 10% price drop produces a 20% demand increase, that is elastic demand.
If it produces a 2% increase, that is inelastic.

**Asymmetric elasticity** means customers respond differently to price drops
vs price increases — they stock up aggressively on discounts but don't
proportionally cut back when prices rise.

### We tested at three levels of granularity

In [ ]:
show('06_price_elasticity.png')

### What the three levels show

**Level 1 — All departments, all stores (r ≈ 0.003–0.13):**
Near-zero correlation. A price change on one product gets averaged across thousands
of other products with no price change — the signal disappears completely.

**Level 2 — FOODS_3 @ CA_3 (r ≈ 0.16–0.19):**
Scoping to one store improves the signal modestly. Still weak.

**Level 3 — Single product FOODS_3_383 @ CA_3:**
- Price drops: **r = 0.553** — a 10% price drop is associated with ~77% demand increase
- Price increases: r = 0.180 — much weaker
- This is **asymmetric elasticity** confirmed

**Why aggregation kills the signal — confounding and reverse causality:**
When you look at the scatter plots at the department level, you see something
counterintuitive: the more prices drop, the *lower* demand is, and the higher
prices go, the *higher* demand is. This looks like the opposite of what economics
would predict — but it is not a real demand relationship. It is a confounding effect.

Here is what is actually happening: Walmart does not randomly discount products.
They mark down products that are **already selling poorly** — slow movers get
price cuts to clear shelf space. So in the data, price drops and low demand
co-occur constantly, but not because price drops *cause* low demand. Both are
caused by the same underlying thing: a product that was not selling well to
begin with. Similarly, high-demand products can sustain higher prices, so
price increases and high demand co-occur for the same reason.

This is called **reverse causality** — instead of price driving demand, demand
is driving price. The direction of causation is backwards from what you would
expect, and it completely masks the true elasticity signal at the aggregate level.

The confounding disappears at the individual product-store level because you are
now tracking *how the same product's demand changes* when *its own price changes*
week over week — controlling for the product's underlying popularity. That is when
the true signal emerges: price drops produce a strong demand response (r=0.553),
and the direction is exactly what economics predicts.

### Modeling decision
Price features must be computed at the `item_id + store_id` level only.
Department-average price features have r < 0.13 and are excluded.
We need **signed** price change features — `price_drop_pct` and `price_increase_pct`
as separate features because the asymmetry is real and meaningful.

---

## 2.5 — Finding 5: Series completeness

Before modeling individual product-store series we need to know how many
series have enough history to be usable.

In [ ]:
show('07_series_completeness.png')

### Key finding

**Only 8.1% of the 30,490 series (2,469 series) have sales in all 64 months.**

The other 91.9% have structural zero gaps — products introduced mid-period,
discontinued, or not stocked at specific stores. The distribution is wide:
some series have only 2 active months, some have all 64.

This is not a data quality issue. It reflects real retail dynamics: products
launch, get rotated, get delisted. The dataset captures that reality.

### Modeling decisions
- **SARIMA** requires a continuous time index — only viable for the 2,469 complete series
- Our **representative series** is selected from these 2,469 as the one closest to
  the median total revenue: **FOODS_3_163 @ CA_3** (~$120/month average)
- **XGBoost** handles incomplete series row by row, but lag features must not span
  structural zero gaps

---
---

# PART 3 — STATIONARITY AND MODEL SELECTION

---

## 3.1 — What is stationarity and why does it matter?

Before fitting SARIMA, we have to satisfy one fundamental requirement.

### Definition

A time series is **stationary** if its statistical properties — mean, variance —
do not change over time. A flat, noisy series with no trend is stationary.
A series with an upward trend is **not stationary** — the mean keeps rising.

### Why SARIMA requires it

SARIMA's mathematical equations assume the series fluctuates around a **stable mean**.
If the mean keeps drifting upward, the model's core assumptions break down and
parameter estimates become unreliable.

### How we fix it: differencing

**Differencing** transforms a non-stationary series into a stationary one by
subtracting past values.

| Type | Operation | What it removes |
|---|---|---|
| **First differencing** (d=1) | Subtract each value from the previous month | Linear trend |
| **Seasonal differencing** (D=1) | Subtract the value from the same month one year ago | Annual drift |

Instead of modeling revenue levels, we model revenue *changes*.

### The ADF test

The **Augmented Dickey-Fuller (ADF) test** is a formal statistical test for stationarity.

- **p-value < 0.05:** stationary — safe to model directly
- **p-value > 0.05:** non-stationary — needs differencing

### Our results

| Series | Raw p-value | Verdict | Fix |
|---|---|---|---|
| Aggregate | 0.318 | Non-stationary | Seasonal differencing (D=1) → p=0.000 ✓ |
| Representative (FOODS_3_163) | 0.0004 | Already stationary | No differencing needed |

**Why does the aggregate need differencing but the individual series doesn't?**
The aggregate series doubled in revenue over 5 years — a strong persistent trend.
The individual product-store series fluctuates around a more stable mean with no
persistent long-term drift.

---

## 3.2 — ACF and PACF: determining the model order

Once we know how much differencing to apply, we need to determine the AR and MA orders.
This is done with ACF and PACF plots.

### Definitions

**ACF (Autocorrelation Function):** Measures how correlated a series is with its own
past values at each lag. A significant spike at lag 3 means "the value from 3 months
ago is correlated with today's value."

**PACF (Partial Autocorrelation Function):** Same as ACF but removes the indirect
effects of intermediate lags. If lag 1 and lag 2 are both significant, PACF tells
you whether lag 2 is significant *on its own* or just because it carries lag 1's information.

### How to read them

| Pattern | What it tells you |
|---|---|
| PACF cuts off sharply after lag p | AR order = p |
| ACF cuts off sharply after lag q | MA order = q |
| ACF decays gradually | AR process — not a sharp cutoff |
| Significant spikes at lag 12 | Seasonal AR (P) or seasonal MA (Q) terms needed |

In [ ]:
show('08_acf_pacf.png')

### Our readings

**Aggregate series (after seasonal differencing):**
- PACF: one significant spike at lag 1, then cuts off → **p = 1 candidate**
- ACF: significant at lags 1–5 but decays gradually → **q = 0** (gradual decay
  is the signature of AR carry-through, not a separate MA process)
- Lag-12: neither ACF nor PACF significant → **P = 0, Q = 0**

**Representative series (no differencing needed):**
- PACF: one significant spike at lag 1 only → **p = 1 candidate**
- ACF: significant at lags 1–2 then cuts off → **q = 1 candidate**
- Lag-12: neither significant → **P = 0, Q = 0**

These are *candidates* — we run a grid search with AIC to confirm the final order.

---

## 3.3 — AIC grid search: confirming the final order

ACF/PACF narrows the search space. **AIC grid search** confirms the final order
by fitting every candidate combination on training data and picking the best.

### What is AIC?

**AIC (Akaike Information Criterion)** is a model selection score that balances
two competing goals:

- **Goodness of fit:** how well does the model explain the training data?
- **Complexity penalty:** each extra parameter is penalized

**Lower AIC = better model.** A model with more parameters must fit meaningfully
better to justify the added complexity.

### Why AIC instead of a validation set?

With only 48 training months, holding out an additional validation fold would cost
too many observations. AIC gives us model selection on the full training set without
touching the test period.

### Final SARIMA orders

| Series | Selected order | Seasonal order | Notes |
|---|---|---|---|
| Aggregate | (2,0,1) | (0,1,1)[12] | AIC extended p to 2; D=1 confirmed |
| Representative | (0,0,1) | (0,1,1)[12] | AR dropped entirely; D=1 selected by AIC despite stationary raw |

---

## 3.4 — SARIMA: how it works

Now we have all the tools to understand the full model.

### SARIMA(p,d,q)(P,D,Q)[m] — the six parameters

| Parameter | Name | What it controls |
|---|---|---|
| **p** | AR order | How many past values directly predict today. AR(2) = this month depends on the last 2 months. |
| **d** | Differencing | How many times to subtract consecutive values to remove trend. |
| **q** | MA order | How many past *forecast errors* correct today's prediction. MA(1) = last month's error adjusts this month's forecast. |
| **P** | Seasonal AR | Same as p but at the seasonal lag (12 months back). |
| **D** | Seasonal differencing | Subtracts the value from the same month one year ago. |
| **Q** | Seasonal MA | Corrects for last year's same-month forecast error. |
| **m** | Seasonal period | Length of one cycle. m=12 for monthly data with annual seasonality. |

### How the components work

**AutoRegression (AR):** The model uses its own past values as predictors.
*"This month's revenue is a weighted sum of the last 2 months plus noise."*
Captures momentum — if revenue has been high recently it tends to stay high.

**Moving Average (MA):** Instead of raw past values, uses past forecast *errors*.
*"If I over-predicted last month by $50K, I correct this month's forecast downward."*
Makes the model self-correcting.

**Integration (differencing):** Removes trend before fitting so the model
works with changes rather than levels.

**Seasonal versions:** All three components repeat at the seasonal lag (lag-12).
Seasonal AR uses revenue from 12 months ago. Seasonal MA corrects for last year's
same-month error.

### The key limitation: mean reversion

SARIMA's AR structure is **mean-reverting by design**. As the forecast horizon extends,
the model gradually pulls predictions back toward the historical mean of the training data.
On a series with a strong upward trend, this means long-horizon forecasts will
systematically underestimate. We will see this clearly in the results.

---

## 3.5 — Prophet: how it works and how it differs from SARIMA

Prophet is a completely different approach built by Meta (Facebook) in 2017.
Instead of one unified equation, it fits three interpretable components separately.

### The Prophet equation

```
y(t) = trend(t) + seasonality(t) + holidays(t) + noise
```

Each component is fitted independently then combined.

### The three components

**Trend:** Models the underlying direction as a **piecewise linear curve**.
Prophet automatically detects **changepoints** — moments where the slope shifts —
and fits a new direction after each one. No manual differencing required.
Crucially: it **does not mean-revert** — it continues the identified slope forward.

**Seasonality:** Models repeating patterns using **Fourier series** — sums of
sine and cosine waves at different frequencies. For monthly data, Prophet fits
annual waves automatically. No ACF/PACF analysis needed.

**Holidays:** Accepts a dataframe of named events with specific dates and window sizes.
Prophet fits a separate effect for each event — directly incorporating the holiday
impacts we quantified in the EDA. This is Prophet's biggest practical advantage over SARIMA.

### Key hyperparameters we tuned

| Parameter | What it controls | Our winning value |
|---|---|---|
| `changepoint_prior_scale` (cps) | Trend flexibility. Low = stiff straight line. High = wiggly trend that bends to follow data. | 0.5 (aggregate), 0.01 (individual/stores) |
| `seasonality_mode` | **Additive** = fixed dollar seasonal swing regardless of trend level. **Multiplicative** = seasonal swing grows as revenue grows. | multiplicative (aggregate), additive (individual) |
| `seasonality_prior_scale` | How strongly seasonal patterns are fitted. | 1.0 across all models |

### How we selected hyperparameters

**Cross-validation on training data only.** Prophet's built-in CV slides the training
window forward, evaluates on a 12-month horizon, and we pick the lowest RMSE.
The test set is never touched during this process — no data leakage.

### SARIMA vs Prophet — head to head

| | SARIMA | Prophet |
|---|---|---|
| **Trend handling** | Differencing removes trend; AR mean-reverts at long horizons | Piecewise linear curve; extrapolates slope forward |
| **Seasonality** | AR/MA terms at seasonal lags | Fourier series — automatic |
| **Holiday effects** | None — completely blind | Explicit holiday component |
| **Parameter selection** | Manual — ADF, ACF/PACF, AIC grid | CV grid search |
| **Strength** | Rigorous on stationary series with clear autocorrelation | Handles trend changes and holidays naturally |
| **Weakness** | Mean reversion at long horizons on trending series | Can overfit with high changepoint flexibility |

---
---

# PART 4 — BASELINES AND THE AGGREGATE SERIES

---

## 4.1 — Train/test split and why it matters

Before any results mean anything, we need a fair evaluation framework.

### The split

| Period | Months | Date range |
|---|---|---|
| **Training** | 48 months | Feb 2011 → Jan 2015 |
| **Test (held out)** | 12 months | Feb 2015 → Jan 2016 |

**Strictly time-based** — no shuffling, no random sampling. Every model is trained
on identical data and evaluated on the identical test period. This is the only valid
way to evaluate time series models — you cannot randomly sample because future values
cannot be used to predict the past.

### The three evaluation metrics

| Metric | Formula | What it measures |
|---|---|---|
| **RMSE** | √(mean of squared errors) | Average error in dollars; penalizes large errors more heavily |
| **MAE** | Mean of absolute errors | Average dollar error per month; simpler, always lower than RMSE |
| **MAPE** | Mean of \|error / actual\| × 100 | Average error as % of actual; unit-free and most interpretable |

**MAPE is our primary metric** because it is comparable across series with very
different revenue scales (aggregate vs individual product-store).

### Baseline models — the floor everything must beat

**Naive (persistence):** Predict next month's value = this month's value.
The simplest possible forecast. If SARIMA cannot beat this, it has no value.

**SMA(3):** 3-month simple moving average. Slightly smoother than naive.

**Interesting finding:** On the aggregate series, naive (8.96% MAPE) actually
*beats* SMA(3) (10.71% MAPE). Why? The series has a strong upward trend.
The most recent month is already the best single estimate. SMA averages in
older, lower values and systematically undershoots the trend.

In [ ]:
show('09_baseline_forecasts.png')

---

## 4.2 — SARIMA results: aggregate series

In [ ]:
show('10_sarima_aggregate.png')

### Results: SARIMA(2,0,1)(0,1,1)[12]

| Metric | Value | vs Naive baseline |
|---|---|---|
| RMSE | $277,220 | −$102,831 ✓ |
| MAE | $252,147 | −$80,094 ✓ |
| MAPE | **6.91%** | −2.05 pp ✓ |

SARIMA beats the naive baseline on every metric. The added complexity is justified.

### What it gets right
The seasonal shape is correct. Early months (Feb–Apr 2015) are very accurate
at 1.7–3.4% error — SARIMA is confident and correct when forecasting close
to the training window.

### What it gets wrong: mean reversion
Errors grow progressively as the horizon extends. By month 12 (January 2016)
the error reaches **10.9%**. The forecast tracks the seasonal pattern correctly
but systematically undershoots actual values at longer horizons.

This is **mean reversion** at work. SARIMA's AR coefficients (ar.L1 = 1.53)
capture strong momentum but the model gradually pulls forecasts back toward
the historical training mean rather than fully continuing the upward trend.
The series kept growing faster than SARIMA expected.

### Why this motivates Prophet
SARIMA sees the trend but does not extrapolate it aggressively enough at
longer horizons. Prophet's piecewise linear trend component is specifically
designed to continue the identified slope — directly addressing this failure.

---

## 4.3 — Prophet results: aggregate series

In [ ]:
show('11_prophet_aggregate_forecast.png')

In [ ]:
show('12_prophet_aggregate_components.png')

### Results: Prophet (cps=0.5, multiplicative)

| Metric | Value | vs SARIMA | vs Naive |
|---|---|---|---|
| RMSE | $209,726 | −$67,494 ✓ | −$170,325 ✓ |
| MAE | $178,567 | −$73,580 ✓ | −$153,674 ✓ |
| MAPE | **5.02%** | −1.89 pp ✓ | −3.94 pp ✓ |

**Best result on the aggregate series.** Prophet beats every prior model on every metric.

### Why Prophet wins: trend continuation
SARIMA's month-12 error: **10.9%**. Prophet's month-12 error: **1.3%**.
Same month, same test data, completely different horizon behavior.
Prophet's piecewise linear trend extrapolates the slope forward and nails
the long-horizon forecast that SARIMA misses entirely.

### The unexpected finding: multiplicative seasonality
The EDA's seasonal decomposition assumed additive seasonality — but the CV
grid search found **multiplicative wins convincingly** (every multiplicative
config outperformed every additive config).

**Why?** Revenue doubled from $2M to $4M per month over 5 years. A seasonal
swing that was $150K in 2011 grew to ~$300K by 2016. Additive seasonality
assumes a fixed dollar amplitude — it systematically underestimates seasonal
peaks in later years. Multiplicative seasonality scales the swing as a
percentage of the trend level, which is the correct specification.

### Reading the component chart
- **Trend:** Clean upward slope — Prophet correctly identifies and extrapolates the growth
- **Yearly seasonality:** The repeating annual pattern — August peaks, February troughs
- **Quarterly:** Finer within-year variation
- **Holiday effects:** Quantified uplift/suppression per event — SuperBowl, Thanksgiving, etc.

### Confidence intervals
Prophet's 95% CI is wider than SARIMA's (±$1.5M by month 12 vs ±$500K).
In multiplicative mode, uncertainty scales with the trend level — as revenue
grows the interval bands grow with it. Honest but wide for operational planning.

---
---

# PART 5 — THE INDIVIDUAL PRODUCT-STORE SERIES

---

## 5.1 — Why individual series are harder

Everything changes when you drop from the platform aggregate to a single
product at a single store.

**The representative series: FOODS_3_163 @ CA_3**
Selected as the series closest to the median revenue among all 2,469 complete
series. Average monthly revenue: ~$120.

| Characteristic | Aggregate series | Individual series |
|---|---|---|
| Monthly revenue | ~$3M | ~$120 |
| Median daily sales | — | 2 units |
| Skewness | Moderate | 11.89 (extreme) |
| Zero rate | Low (aggregated away) | 68.2% |
| Naive MAPE | 8.96% | 55.29% |

The law of large numbers smooths out individual series noise at the aggregate level.
At the individual level, one promotion, one stockout, or one price change can
double or halve a month's revenue. The baseline is dramatically higher.

---

## 5.2 — SARIMA results: individual series

In [ ]:
show('10_sarima_representative.png')

### Results: SARIMA(0,0,1)(0,1,1)[12]

| Metric | Value | vs SMA(3) baseline |
|---|---|---|
| RMSE | $54.41 | −$31.25 ✓ |
| MAE | $37.39 | −$39.92 ✓ |
| MAPE | **22.22%** | −23.58 pp ✓ |

SARIMA nearly halves the baseline error — a 51% relative improvement.
Capturing the seasonal structure is real and exploitable even at the individual series level.

### Notable: the model order changed completely from the aggregate

The representative series selected **p=0** — no AR term at all. On a noisy individual
series, last month's actual sales value adds no direct predictive power. Month-to-month
sales are too erratic to carry useful signal forward. The model relies entirely on
error correction (MA terms) rather than momentum (AR terms).

### The three failure months
April 2015 (54.9%), May 2015 (47.3%), January 2016 (55.6%) — errors exceeding 40%.
These are likely genuine demand spikes driven by a price change or SNAP event.
No statistical model can anticipate these from historical patterns alone.

---

## 5.3 — Prophet results: individual series

In [ ]:
show('11_prophet_representative_forecast.png')

### Results: Prophet (cps=0.01, additive)

| Metric | Value | vs SMA(3) baseline |
|---|---|---|
| RMSE | $52.80 | −$32.86 ✓ |
| MAE | $39.68 | −$37.63 ✓ |
| MAPE | **24.25%** | −21.55 pp ✓ |

### SARIMA vs Prophet: a statistical tie

SARIMA wins on MAPE (22.22% vs 24.25%). Prophet wins on RMSE ($52.80 vs $54.41).
Neither is meaningfully better than the other.

### Why additive seasonality wins here (opposite of aggregate)
The individual series has intermittent, zero-inflated demand. In multiplicative mode,
Prophet scales seasonal swings as a percentage of the trend — on a series where some
months are near zero, this produces erratic, unstable forecasts. Additive seasonality
adds a fixed dollar swing regardless of level, which is more stable for sparse series.

### Why cps=0.01 (stiff trend) wins here (opposite of aggregate)
The aggregate series had a strong, consistent upward trend worth extrapolating (cps=0.5).
The individual series fluctuates around a more stable mean — a flexible trend would
overfit to noise. A stiff trend line (cps=0.01) correctly ignores the month-to-month
spikes and models the underlying baseline.

### The same three months fail
April 2015 (58.5%), May 2015 (49.8%), January 2016 (40.0%) — the exact same months
that SARIMA missed, at similar magnitudes. This is not a coincidence.

---

## 5.4 — The signal ceiling

Two completely different model families. Same failure months. Similar magnitudes.
This is the most important finding of the project.

In [ ]:
show('13_signal_ceiling.png')

### What is the signal ceiling?

The **signal ceiling** is the maximum accuracy achievable using only historical
time series data — regardless of how sophisticated the model is.

Both SARIMA and Prophet have access to exactly the same information: past revenue values
and calendar structure. They fail on the same months because those months were driven
by external events that neither model can observe.

### What caused the shared failure months?

| Month | Likely cause | EDA evidence |
|---|---|---|
| April 2015 | Price drop event | r=0.553 for drops at product-store level — 10% drop → ~77% demand surge |
| May 2015 | Continuation of price effect | Asymmetric elasticity means stock-up behavior persists |
| January 2016 | SNAP distribution timing or post-holiday restocking | CA SNAP uplift +10.3% — timing shift can move a month's demand |

### Why no univariate model can fix this

A univariate model reads only the time series itself. The price history, the SNAP
calendar, local promotions — these signals exist in the dataset but are not in the
time series. No amount of parameter tuning changes what data a model can see.

**The fix is not a better statistical model. It is different inputs.**

---
---

# PART 6 — STORE-LEVEL RESULTS

---

## 6.1 — Why model stores independently?

Store-level forecasts sit between the platform aggregate and the individual
product-store — enabling store managers to plan staffing, shelf space, and
local procurement independently of platform-wide trends.

**We fit three independent Prophet models** — one per store — rather than a single
shared model. Each store has a different revenue baseline, different trend slope,
and potentially different seasonal pattern. Forcing them into one model requires
store dummies and interaction terms that add complexity without adding signal.

**Each store gets its own CV grid search.** The aggregate winning parameters
(cps=0.5, multiplicative) are not assumed to transfer.

### The three stores

| Store | Revenue share | Monthly revenue (train avg) | CV winner |
|---|---|---|---|
| CA_3 | 17.1% | ~$560K/month | cps=0.01, multiplicative |
| CA_1 | 12.0% | ~$390K/month | cps=0.01, multiplicative |
| TX_2 | 10.9% | ~$313K/month | cps=0.05, additive |

---

## 6.2 — CA_1: the best result in the project

In [ ]:
show('14_store_ca1_forecast.png')

### Results: CA_1 — Prophet (cps=0.01, multiplicative)

| Metric | Value |
|---|---|
| RMSE | $13,104 |
| MAE | $11,313 |
| **MAPE** | **2.66%** |

**2.66% MAPE — the best result across any series in this project at any hierarchy level.**
Every month lands within 5.1%. The final month (January 2016) is just 0.2% error.

CA_1 has the smoothest, most consistent revenue trajectory of the three stores —
exactly the conditions where Prophet's piecewise linear trend excels. The model
nails both the trend slope and seasonal shape cleanly.

---

## 6.3 — CA_3: good but with a directional bias

In [ ]:
show('15_store_ca3_forecast.png')

### Results: CA_3 — Prophet (cps=0.01, multiplicative)

| Metric | Value |
|---|---|
| RMSE | $39,078 |
| MAE | $36,472 |
| **MAPE** | **6.23%** |

Good result, but with a consistent directional bias — the model **overshoots**
almost every month. This is the mirror image of TX_2's undershoot.

CA_3's actual revenue growth decelerated slightly in the test period relative to
the training trend. Prophet, trained on the faster growth rate, projects higher
than actuals. November 2015 is the largest miss at 11.8%, likely reflecting
softer-than-expected pre-holiday demand. Despite the bias, errors stay within an
operationally acceptable range.

---

## 6.4 — TX_2: the univariate ceiling

In [ ]:
show('16_store_tx2_forecast.png')

### Results: TX_2 — Prophet (cps=0.05, additive)

| Metric | Value |
|---|---|
| RMSE | $56,856 |
| MAE | $55,343 |
| **MAPE** | **15.31%** |

TX_2 undershoots every single month. The gap widens progressively —
February 2015 error is 12.9%, January 2016 is 21.7%. This compounding
pattern is the signature of a trend slope underestimate: the model learned
a growth rate from training that is slightly slower than what TX_2 actually
delivered in the test period, and the gap accumulates month by month.

### Why it cannot be fixed with Prophet alone

TX_2's year-over-year growth accelerated from **8.2%** during training to
**11.6%** in the test period. A linear trend extrapolation from training
lands within $550/month of the actual test mean — so the trend itself is
correct. Prophet is just not picking up the acceleration.

This is a fundamental limitation of univariate forecasting on a series with
accelerating growth — not a tuning failure.

### Why XGBoost will help TX_2 most

TX_2's SNAP uplift is the strongest signal the EDA identified:
TX FOODS showed **+17.2% revenue on SNAP days**. Prophet sees none of this.
An XGBoost model with `snap_TX` as an explicit feature directly encodes the
demand driver that is pushing TX_2 above its projected trajectory.
TX_2 is the store where the ML layer will add the most value.

---
---

# PART 7 — MASTER COMPARISON AND CONCLUSIONS

---

## 7.1 — Master model comparison

In [ ]:
show('17_mape_comparison_bar.png')

All models evaluated on the same held-out 12-month test period (Feb 2015 → Jan 2016).
Bold = winner per series.

---

### Platform aggregate revenue

| Model | RMSE ($) | MAE ($) | MAPE |
|---|---|---|---|
| Naive | 380,051 | 332,241 | 8.96% |
| SMA(3) | 443,310 | 396,285 | 10.71% |
| SARIMA(2,0,1)(0,1,1)[12] | 277,220 | 252,147 | 6.91% |
| **Prophet — cps=0.5, multiplicative** | **209,726** | **178,567** | **5.02%** |

---

### Individual product-store — FOODS_3_163 @ CA_3

| Model | RMSE ($) | MAE ($) | MAPE |
|---|---|---|---|
| Naive | 98.48 | 91.31 | 55.29% |
| SMA(3) | 85.66 | 77.31 | 45.80% |
| **SARIMA(0,0,1)(0,1,1)[12]** | **54.41** | **37.39** | **22.22%** |
| Prophet — cps=0.01, additive | 52.80 | 39.68 | 24.25% |

*SARIMA wins on MAPE, Prophet wins on RMSE — statistical tie.*

---

### Store level — top 3 stores by revenue

| Store | Revenue share | Model | RMSE ($) | MAE ($) | MAPE |
|---|---|---|---|---|---|
| **CA_1** | 12.0% | **Prophet — cps=0.01, multiplicative** | **13,104** | **11,313** | **2.66%** |
| CA_3 | 17.1% | Prophet — cps=0.01, multiplicative | 39,078 | 36,472 | 6.23% |
| TX_2 | 10.9% | Prophet — cps=0.05, additive | 56,856 | 55,343 | 15.31% |

*TX_2 represents the univariate ceiling — systematic undershoot driven by
unobserved SNAP and price signals. XGBoost target: < 10%.*

---

## 7.2 — Three answers to the business question

Returning to the original question with concrete numbers.

**Q: How many units will store X sell next month?**

| Level | Best model | MAPE | Practical use |
|---|---|---|---|
| Platform aggregate | Prophet | 5.02% | Budget planning, category procurement, capacity |
| Store (CA_1) | Prophet | 2.66% | Store staffing, shelf space, local procurement |
| Store (CA_3) | Prophet | 6.23% | Store-level planning with directional bias noted |
| Store (TX_2) | Prophet | 15.31% | Indicative only — needs external features |
| Individual product-store | SARIMA/Prophet | 22.22% | Anomaly flagging, not precise unit ordering |

**Q: Are sudden sales spikes genuine demand or anomalies?**

The shared failure months — April/May 2015 and January 2016 — are confirmed anomalies.
Both SARIMA and Prophet flag them by failing to forecast them from historical patterns.
These are the highest-priority targets for anomaly detection in the next phase.

---

## 7.3 — What XGBoost adds: closing the three gaps

Statistical models have hit their ceiling. The ML layer is not about winning
a benchmark — it is about closing three specific gaps.

### Gap 1 — External features

| Feature | Source | EDA evidence |
|---|---|---|
| `sell_price` | sell_prices.csv | Price level directly affects demand |
| `price_change_pct` | Derived | r=0.553 for drops at product-store level |
| `price_drop_pct` | Derived | Asymmetric: drops produce ~77% demand surge per 10% drop |
| `snap_CA / snap_TX / snap_WI` | calendar.csv | +10–32% SNAP uplift confirmed by state |
| `is_event_day` | calendar.csv | SuperBowl +18.9%, Labor Day +19.6% |

### Gap 2 — Lag features encode recent momentum

SARIMA's AR terms are estimated from the full training history and cannot
adapt to recent acceleration. XGBoost lag features are computed from recent actuals:

| Feature | What it captures |
|---|---|
| `lag_7` | Sales from one week ago — 7-day cycle |
| `lag_28` | Sales from four weeks ago — 4-week cycle |
| `rolling_mean_7` | Smoothed 7-day trend |
| `rolling_mean_28` | Medium-term baseline |

### Gap 3 — Scale across the full hierarchy

One SARIMA per series caps at 2,469 complete series. XGBoost trains a single model
across all 30,490 simultaneously, using `store_id` and `dept_id` encodings to capture
individual baselines. Evaluated via **walk-forward cross-validation** to prevent leakage.

**Definition — walk-forward CV:** Train on months 1–24, evaluate on month 25.
Then train on months 1–25, evaluate on month 26. Repeat forward. Never trains
on data that comes after the evaluation period — the only valid CV for time series.

### Expected performance targets

| Series | Statistical ceiling | XGBoost target | Primary driver |
|---|---|---|---|
| Aggregate | 5.02% MAPE | Marginal improvement | Prophet already handles trend |
| Individual (FOODS_3_163) | 22.22% MAPE | < 15% | Price and SNAP features |
| TX_2 store | 15.31% MAPE | < 10% | SNAP_TX + lag features |

---

## 7.4 — Limitations

| Limitation | Impact |
|---|---|
| Revenue is derived (units × price) — not directly observed | Minor gaps where price records are missing; filled with zero |
| Only 8.1% of series have complete 64-month histories | SARIMA viable for 2,469 of 30,490 series only |
| One representative series at median revenue | 22% MAPE is a point estimate — varies across the full distribution |
| All statistical models operate on monthly data | XGBoost targets daily — zero-inflation makes it substantially harder |
| Training data ends April 2016 | No COVID, no e-commerce era — patterns may not transfer to modern Walmart |
| Univariate models cannot capture cross-series relationships | SNAP demand in FOODS simultaneously suppresses HOBBIES — missed entirely |

---

## 7.5 — Project summary

| What we built | An end-to-end retail demand forecasting pipeline on 5.25 years of real Walmart data |
|---|---|
| **Dataset** | 30,490 product-store series, 58M rows, 3 states, 10 stores |
| **EDA findings** | Zero-inflation (68.2%), SNAP uplift (+10–32% by state), asymmetric price elasticity (r=0.55), 8.1% complete series |
| **Models** | Naive, SMA, SARIMA, Prophet — evaluated at 3 hierarchy levels |
| **Best aggregate result** | Prophet 5.02% MAPE — beats naive by 3.94 pp |
| **Best store result** | CA_1 Prophet 2.66% MAPE — best in project |
| **Individual ceiling** | ~22% MAPE — SARIMA and Prophet statistically tied |
| **Key finding** | Signal ceiling confirmed — same 3 months fail across both model families |
| **Next step** | XGBoost with price, SNAP, and lag features — targets the exact gaps statistical models cannot close |